In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def get_article_links(page_url):
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    
    try:
        response = requests.get(page_url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')
        articles = []
        
        for div in soup.find_all('div', class_='m'):
            link = div.find('a')
            if link and link.get('href'):
                href = link['href']
                if '/video/' in href:
                    continue
                
                full_url = f"https://www.foxnews.com{href}"
                img = link.find('img')
                title = img['alt'].strip() if img and img.get('alt') else "No title"
                
                articles.append({'title': title, 'url': full_url})
        
        return articles
    
    except Exception as e:
        print(f"Error: {e}")
        return []

def crawl_foxnews(pages=20):
    all_articles = []
    base_url = "https://www.foxnews.com/category/politics/judiciary/abortion"
    
    for page in range(1, pages + 1):
        print(f"Crawling page {page}...")
        
        page_url = f"{base_url}?page={page}" if page > 1 else base_url
        articles = get_article_links(page_url)
        
        if articles:
            all_articles.extend(articles)
            print(f"Found {len(articles)} articles")
        
        time.sleep(1)
    
    return all_articles

def save_to_excel(articles, filename="foxnews_articles.xlsx"):
    if not articles:
        print("No articles found")
        return
    
    df = pd.DataFrame(articles)
    df.to_excel(filename, index=False)
    print(f"Saved {len(articles)} articles to {filename}")

def main():
    print("Starting Fox News crawler...")
    articles = crawl_foxnews(pages=20)
    
    if articles:
        print(f"Total articles: {len(articles)}")
        save_to_excel(articles)
    else:
        print("No articles found")

if __name__ == "__main__":
    main()

Starting Fox News crawler...
Crawling page 1...
Found 28 articles
Crawling page 2...
Found 26 articles
Crawling page 3...
Found 26 articles
Crawling page 4...
Found 30 articles
Crawling page 5...
Found 26 articles
Crawling page 6...
Found 25 articles
Crawling page 7...
Found 29 articles
Crawling page 8...
Found 23 articles
Crawling page 9...
Found 21 articles
Crawling page 10...
Found 26 articles
Crawling page 11...
Found 23 articles
Crawling page 12...
Found 25 articles
Crawling page 13...
Found 25 articles
Crawling page 14...
Found 24 articles
Crawling page 15...
Found 28 articles
Crawling page 16...
Found 21 articles
Crawling page 17...
Found 24 articles
Crawling page 18...
Found 27 articles
Crawling page 19...
Found 28 articles
Crawling page 20...
Found 29 articles
Total articles: 514
Saved 514 articles to foxnews_articles.xlsx


In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

def simple_foxnews_crawler(excel_file):
    df = pd.read_excel(excel_file)
    results = []
    
    for index, row in df.iterrows():
        url = row['url']
        title = row.get('title', '')
        
        print(f"Processing: {title}")
        
        try:
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.content, 'html.parser')
            
            article_body = soup.find('div', class_='article-body')
            
            # Extract time and content
            time_element = soup.find('time')
            publish_time = time_element.get_text(strip=True) if time_element else "Time not found"
            
            if article_body:
                first_speakable = article_body.find('p', class_='speakable')
                
                if first_speakable:
                    content_paragraphs = []
                    current_element = first_speakable
                    
                    while current_element:
                        if current_element.name == 'p':
                            text = current_element.get_text(strip=True)
                            if (len(text) > 30 and 
                                not text.startswith('CLICK HERE') and 
                                not text.startswith('GET THE') and
                                'DOWNLOAD' not in text):
                                content_paragraphs.append(text)
                        
                        current_element = current_element.find_next_sibling()
                        
                    content = '\n'.join(content_paragraphs)
                else:
                    content = "No speakable paragraphs found"
            else:
                content = "No article content found"
                
            results.append({
                'title': title,
                'url': url,
                'publish_time': publish_time,
                'content': content
            })
            
        except Exception as e:
            results.append({
                'title': title,
                'url': url,
                'publish_time': "Extraction failed",
                'content': f"Error: {str(e)}"
            })
        
        time.sleep(1)
    
    result_df = pd.DataFrame(results)
    result_df.to_excel('foxnews_articles_crawled.xlsx', index=False)
    print("Completed!")

simple_foxnews_crawler("https://raw.githubusercontent.com/FNXNRNJOYCE/HKU/main/7005/foxnews_articles.xlsx")

Processing: Pro-life influencer attacked in NYC files lawsuit after DA Alvin Bragg drops case
Processing: Missouri attorney general takes new legal aim at mail-order abortion pills over safety concerns
Processing: Pro-life pregnancy centers see client increase after Supreme Court decision: study
Processing: Northwestern Student Committee gets donated ‘reproductive vending machine’ with free Plan B, condoms, Narcan
Processing: Record 40% of young women want to flee US: poll
Processing: Florida cites mafia law, hits Planned Parenthood with suit over claim abortion pill 'safer than Tylenol'
Processing: Florida man facing death penalty for killing 18-year-old girlfriend, unborn child after she refused abortion
Processing: Republican senators blast FDA for expanding abortion pill access
Processing: Riley Gaines and Ocasio-Cortez trade barbs as conservatives defend the former swimmer in ongoing dispute
Processing: Newsom bails out Planned Parenthood with $140M to keep 100 clinics open after 